## NHANES Metabolic Syndrome — Conformance Inference

Adapts the conformal inference pipeline to the metabolic syndrome dataset.
Uses pre-generated train/test splits and iterates over the same feature combinations as `nn_ns_3.ipynb`.

In [5]:
import sys
sys.path.append("..")

import os
import pandas as pd
import random
import numpy as np
import torch

from data_real import load_data
from train import train_conformance_inference
from metrics import compute_classification_metrics
from utils import get_device

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, roc_auc_score, log_loss
)

device = get_device()

random.seed(0)
torch.manual_seed(0)
np.random.seed(0)

In [6]:
n_folds = 5
n_epochs = 300
batch_size = 64
learning_rate = 0.001
weight_decay = 0.01
dropout = 0.2
hidden_sizes_bc = [32, 16, 8]
hidden_sizes_rr = [16, 8]
alpha = 0.2
patience_classification = 4
min_delta_classification = 1e-4
patience_regression = 4
min_delta_regression = 1e-4
standardize = True

In [7]:
def evaluate_on_test(ci_result, test_data):
    """Evalúa el resultado de conformance inference en test y retorna métricas detalladas."""
    y_pred_proba = ci_result['p']
    y_pred_bin = (y_pred_proba >= 0.5).astype(int)
    y_true = np.ravel(test_data['y'])

    accuracy = accuracy_score(y_true, y_pred_bin)
    auc_score = roc_auc_score(y_true, y_pred_proba)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_bin, average='weighted'
    )
    cm = confusion_matrix(y_true, y_pred_bin)
    ce = log_loss(y_true, y_pred_proba)

    return {
        'accuracy': accuracy,
        'auc': auc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'ce': ce,
        'cm': cm,
    }

In [ ]:
comb_1 = ['RIDAGEYR', 'RIAGENDR']
comb_2 = ['BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS']
comb_3 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS']
comb_4 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSTR']
comb_5 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR', 'LBXGH']
comb_6 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR', 'LBXGLU']
comb_7 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR', 'LBXGLU', 'LBXGH']
combinations = [comb_1, comb_2, comb_3, comb_4, comb_5, comb_6, comb_7]
# combinations = [comb_4]

# Load pre-generated splits
splits_data = np.load("../data/splits.npz")
train_seqn = splits_data['train_seqn']
test_seqn  = splits_data['test_seqn']

os.makedirs("models", exist_ok=True)

results_list = []

for i, comb in enumerate(combinations):
    print(f"[{i+1}/{len(combinations)}] Procesando combinación {i}: {len(comb)} features")

    file_real = os.path.join("../data", "nhanes_with_metabolic_syndrome_adults.csv")
    data = load_data(file_real, comb, 'metabolic_syndrome', 'WTMEC2YR')

    seqn = data['seqn']
    train_split = np.where(np.isin(seqn, train_seqn))[0]
    test_split  = np.where(np.isin(seqn, test_seqn))[0]

    train_data = {
        'seqn': data['seqn'][train_split],
        'x': data['x'][train_split],
        'y': data['y'][train_split],
        'w': data['w'][train_split],
        'z': data['z'][train_split]
    }

    test_data = {
        'seqn': data['seqn'][test_split],
        'x': data['x'][test_split],
        'o': data['x'][test_split],
        'y': data['y'][test_split],
        'w': data['w'][test_split],
        'z': data['z'][test_split]
    }

    scaler = StandardScaler()
    if standardize:
        train_data['x'] = scaler.fit_transform(train_data['x'])
        test_data['x'] = scaler.transform(test_data['x'])

    predictor = train_conformance_inference(
        data=train_data,
        n_folds=n_folds,
        n_epochs=n_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        alpha=alpha,
        patience_classification=patience_classification,
        min_delta_classification=min_delta_classification,
        patience_regression=patience_regression,
        min_delta_regression=min_delta_regression,
        dropout=dropout,
        hidden_sizes_bc=hidden_sizes_bc,
        hidden_sizes_rr=hidden_sizes_rr
    )

    # Move models to CPU and save bundle (predictor + scaler + colnames)
    predictor._model_ce = predictor._model_ce.cpu()
    predictor._model_rr = predictor._model_rr.cpu()
    torch.save(
        {'predictor': predictor, 'scaler': scaler, 'colnames': data['colnames']},
        f"models/predictor_comb_{i}.pt"
    )

    ci_result, ci_correct = predictor.classify(test_data)
    test_metrics = evaluate_on_test(ci_result, test_data)

    pd.reset_option('display.max_rows')
    pd.set_option('display.max_rows', None)

    df = pd.DataFrame(test_data['o'])
    df.columns = data['colnames']
    df['Seqn'] = test_data['seqn']
    df['conditions'] = ci_result['conditions']
    df['quantiles'] = ci_result['quantiles']
    df['score_pos'] = ci_result['score_pos']
    df['score_neg'] = ci_result['score_neg']
    df['Prob.'] = ci_result['p']
    df['Output'] = ci_result['y']
    df['CI'] = ci_result['ci']
    df['Target'] = np.int32(test_data['y'])

    df.to_csv("conformal_ns" + str(i) + ".csv", index=False)

    result_row = {
        'Combination': i,
        'N_Features': len(comb),
        'Features': ', '.join(comb),
        'Test_AUC': float(test_metrics['auc']),
        'Test_Accuracy': float(test_metrics['accuracy']),
        'Test_Precision': float(test_metrics['precision']),
        'Test_Recall': float(test_metrics['recall']),
        'Test_F1': float(test_metrics['f1']),
        'Test_CE': float(test_metrics['ce']),
        'CI_Correct': float(ci_correct)
    }
    results_list.append(result_row)

    print(f"  Test  - AUC: {test_metrics['auc']:.4f}, Acc: {test_metrics['accuracy']:.4f}, F1: {test_metrics['f1']:.4f}, CE: {test_metrics['ce']:.4f}, CI Correct: {ci_correct:.4f}")
    print()

results_df = pd.DataFrame(results_list)

print("\n" + "="*120)
print("RESULTADOS COMPARATIVOS")
print("="*120)
display(results_df[[
    'Combination', 'N_Features',
    'Test_AUC', 'Test_Accuracy',
    'Test_Recall', 'Test_Precision', 'Test_F1',
    'Test_CE', 'CI_Correct'
]].style.format({
    'Test_AUC': '{:.4f}', 'Test_Accuracy': '{:.4f}',
    'Test_Recall': '{:.4f}', 'Test_Precision': '{:.4f}',
    'Test_F1': '{:.4f}', 'Test_CE': '{:.4f}',
    'CI_Correct': '{:.4f}'
}).background_gradient(subset=['Test_AUC'], cmap='RdYlGn'))

results_df.to_csv('comparison_results_conformance_ns.csv', index=False)
print(f"\nResultados guardados en 'comparison_results_conformance_ns.csv'")